In [ ]:
import base64
import gzip
import json
import pickle
import pyrender
import math
import numpy as np
import random
import secrets
import trimesh
import voxeloo
import zlib

from PIL import Image
from dataclasses import dataclass
from functools import cache
from matplotlib import pyplot as plt
from scipy.ndimage import (
    distance_transform_edt as edt,
    gaussian_filter as blur, 
)
from tqdm.notebook import tqdm
from typing import Any, List, Optional, Dict, Tuple
from scipy import ndimage

# Needed for tiling properties.
from perlin_numpy import generate_perlin_noise_2d

### Visualization Routines

In [ ]:
def heatmap_to_image(heat):
    return Image.fromarray(np.uint8(heat * 255) , "L")

In [ ]:
def visualize_voxels(voxels, wireframe=False):
    if voxels.dtype == bool:
        voxels = np.array([0, 0xffffffff])[voxels.astype(int)]
    
    mesh = voxeloo.voxels.voxels_to_mesh(voxels)
    
    # Convert the mesh into the trimesh format.
    tm = trimesh.Trimesh(
        vertices=mesh.vertices[:, 0:3],
        faces=mesh.triangles,
    )
    
    # Add the vertex colors.
    tm.visual.face_colors = mesh.vertices[mesh.triangles[:, 0], 3:6]
    
    scene = pyrender.Scene(ambient_light=[0.2, 0.2, 0.2, 1.0])
    scene.add(
        pyrender.Mesh.from_trimesh(
            tm,
            smooth=False,
            wireframe=wireframe,
        )
    )
    pyrender.Viewer(
        scene,
        use_raymond_lighting=True,
    )

### Noise Routines

In [ ]:
def seed_hash(key):
    return np.uint32(zlib.adler32(str(key).encode("utf-8")))

In [ ]:
def domain(shape, scale):
    coords = np.meshgrid(
        *[np.arange(dim) for dim in shape],
        indexing="ij",
    )
    return tuple(coord / scale for coord in coords)[::-1]

In [ ]:
@cache 
def gen(seed=None):
    return voxeloo.noise.SimplexNoise(seed_hash(seed or 1))
    
def noise(coords, seed=None, mod=None):
    return gen(seed).noise(coords.reshape(-1, coords.shape[-1])).reshape(*coords.shape[0:-1])

In [ ]:
def explicit_noise(shape, period, weights, lacunarity=2.0, transform=None, seed=None):
    ret = np.zeros(shape, dtype=float)
    for i, w in enumerate(weights):
        coords = np.stack(domain(shape, period / lacunarity**i), axis=-1)
        if transform:
            coords = transform(coords)
        ret += w * noise(coords, seed)
    return ret

In [ ]:
def periodic_noise(shape, period, weights, lacunarity=2, seed=None):
    np.random.seed(seed_hash(seed))
    ret = np.zeros(shape, dtype=float)
    res = np.array(shape) // period
    for i, w in enumerate(weights):
        octave = generate_perlin_noise_2d(shape, res, tileable=[True, True])
        ret += w * octave
        res *= lacunarity
    return ret   

In [ ]:
def normalize(nm):
    return (nm - nm.min()) / (nm.max() - nm.min())

### Define voxels types

In [ ]:
@dataclass
class TerrainVoxel:
    name: str
    index: int
    color: int

In [ ]:
def to_flora_id(terrain_id):
    return (1 << 24) | (terrain_id & 0xffffff)

# These IDs match the terrain IDs in: src/galois/js/assets/blocks.ts
terrain = [
    # Blocks
    TerrainVoxel("none", 0, 0x00000000),
    TerrainVoxel("basalt", 8, 0x332c2bff),
    TerrainVoxel("bedrock", 6, 0x4e4a54ff),
    TerrainVoxel("birch_log", 12, 0xd4cdc5ff),
    TerrainVoxel("clay", 17, 0x6e554fff),
    TerrainVoxel("coal_ore", 19, 0x282e2aff),
    TerrainVoxel("cobblestone", 5, 0x524535ff),
    TerrainVoxel("diamond_ore", 23, 0x89c5d9ff),
    TerrainVoxel("dirt", 2, 0x543919ff),
    TerrainVoxel("gold_ore", 11, 0xdecb52ff),
    TerrainVoxel("granite", 13, 0x636d70ff),
    TerrainVoxel("grass", 1, 0x319436ff),
    TerrainVoxel("gravel", 36, 0xab9865ff),
    TerrainVoxel("hay", 35, 0xa39f33ff),
    TerrainVoxel("limestone", 9, 0xc3c4b1ff),
    TerrainVoxel("moss", 40, 0x30784eff),
    TerrainVoxel("neptunium_ore", 30, 0x1d4030ff),
    TerrainVoxel("oak_log", 3, 0x4d442cff),
    TerrainVoxel("pumpkin", 10, 0xc78e08ff),
    TerrainVoxel("quartzite", 7, 0xa89c8aff),
    TerrainVoxel("rubber_log", 14, 0x94570cff),
    TerrainVoxel("silver_ore", 25, 0xd8e3e3ff),
    TerrainVoxel("stone", 4, 0xc1c9c8ff),
    TerrainVoxel("snow", 37, 0xf5f5f5ff),
    
    # Flora
    TerrainVoxel("oak_leaf", to_flora_id(1), 0x234a2dff),
    TerrainVoxel("birch_leaf", to_flora_id(2), 0x586b23ff),
    TerrainVoxel("rubber_leaf", to_flora_id(3), 0x176e47ff),
    TerrainVoxel("switch_grass", to_flora_id(4), 0x1ad94aff),
    TerrainVoxel("azalea_flower", to_flora_id(5), 0xb84f9dff),
    TerrainVoxel("bell_flower", to_flora_id(6), 0x5474a8ff),
    TerrainVoxel("dandelion_flower", to_flora_id(7), 0xadb020ff),
    TerrainVoxel("daylily_flower", to_flora_id(8), 0xbf5a17ff),
    TerrainVoxel("lilac_flower", to_flora_id(9), 0x742fb5ff),
    TerrainVoxel("rose_flower", to_flora_id(10), 0xb51200ff),
    TerrainVoxel("cotton_flower", to_flora_id(11), 0xd1c9c7ff),
    TerrainVoxel("hemp_bush", to_flora_id(12), 0x18381eff),
    TerrainVoxel("red_mushroom", to_flora_id(13), 0x82283fff),
]

terrain_index = {tv.name: tv.index for tv in terrain}

terrain_color = np.zeros(max(tv.index for tv in terrain) + 1, dtype=np.uint32)
for tv in terrain:
    terrain_color[tv.index] = tv.color

### Generate some trees

In [ ]:
def march_voxels(pos: Tuple, dir: Tuple, fn):
    x, y, z = pos
    r, s, t = dir
    
    sx = -1 if r < 0 else 1
    sy = -1 if s < 0 else 1
    sz = -1 if t < 0 else 1
    
    norm = (r * r + s * s + t * t)**0.5
    dx = norm / abs(r) if r != 0 else float("inf")
    dy = norm / abs(s) if s != 0 else float("inf")
    dz = norm / abs(t) if t != 0 else float("inf")

    ix = math.floor(x)
    iy = math.floor(y)
    iz = math.floor(z)
    
    lx = dx * (x - ix if sx == -1 else 1 + ix - x)
    ly = dy * (y - iy if sy == -1 else 1 + iy - y)
    lz = dz * (z - iz if sz == -1 else 1 + iz - z)

    d = 0
    while not fn(ix, iy, iz, d):
        if lx <= ly and lx <= lz:
            d = lx
            ix += sx
            lx += dx
        elif ly <= lz:
            d = ly
            iy += sy
            ly += dy

        else:
            d = lz
            iz += sz
            lz += dz

In [ ]:
def draw_branch(voxels, pos, dir, distance, voxel):
    final_pos = np.array(pos).astype(int)
    def __inner(ix, iy, iz, d):
        voxels[ix, iy, iz] = voxel
        final_pos[:] = (ix, iy, iz)
        return d >= distance
    march_voxels(pos, dir, __inner)
    return final_pos

In [ ]:
def branch_vector(t, dy):
    x = np.array([
        math.sin(t),
        dy,
        math.cos(t),
    ])
    return x / np.linalg.norm(x)

In [ ]:
def decay(mask, noise=0.1, amount=0.6):
    clip = noise * (2.0 * np.random.random(mask.shape) - 1.0)
    edge = clip + blur(mask.astype(float), 1) < amount
    mask[edge] = False

In [ ]:
trees = []
tree_kinds = []

#### Define oak trees

In [ ]:
heights = [
    *[7] * 8,
    *[9] * 6,
    *[11] * 7,
    *[12] * 7,
    *[15] * 4,
    *[16] * 4,
]

print(f"Generating {len(heights)} trees...")
for h in heights:    
    tree = np.zeros((16, 32, 16), dtype=int)
    mask = np.zeros((16, 32, 16), dtype=bool)
    
    
    log = terrain_index["oak_log"]
    leaf = terrain_index["oak_leaf"]
    
    iz, iy, ix = draw_branch(tree, (8.5, 0.5, 8.5), (0, 1, 0), h, log)
    mask[iz, iy, ix] = 1
    
    t = 2.0 * math.pi * random.random()
    
    for j in np.linspace(4, 0.8 * h, 2 * h // 3):
        t = t + 0.75 * math.pi
        l = min(0.4 * h - 0.4 * abs(j - 0.7 * h), 7)
        s = j / (h - 2) + 0.1 * np.random.randn()
        
        ray = branch_vector(t, s)
        iz, iy, ix = draw_branch(tree, (8.5, j, 8.5), ray, l, log)
        mask[iz, iy, ix] = 1
    
    r = 0.2 * h if h <= 10 else 0.1 * h 
    
    mask = blur(mask.astype(float), r) > 0.004
    decay(mask, 0.3, 0.5) # Decay some random leafs on the surface
    mask[:, :3, :] = False  # No leaves in bottom 3 voxels
    
    tree[np.logical_and(mask, tree == 0)] = leaf
    
    trees.append(tree)
    tree_kinds.append("oak")

In [ ]:
#visualize_voxels(terrain_color[random.choice(trees)])
visualize_voxels(terrain_color[trees[0]])

#### Define birch trees

In [ ]:
heights = [
    *[5] * 8,
    *[6] * 10,
    *[7] * 10,
    *[8] * 8,
]

print(f"Generating {len(heights)} trees...")
for h in heights:    
    tree = np.zeros((16, 32, 16), dtype=int)
    mask = np.zeros((16, 32, 16), dtype=bool)
    
    log = terrain_index["birch_log"]
    leaf = terrain_index["birch_leaf"]
    
    t = 2.0 * math.pi * np.random.uniform()
    s = 2.0 + 0.1 * np.random.randn()
    
    draw_branch(tree, (8.5, 0, 8.5), (0, 1, 0), 2, log)

    iz, _, ix = (0.5 * h * branch_vector(t + 0.5 * math.pi, 0.0)).astype(int)
    mask[8 + iz, h + 3, 8 + ix] = 1
    iz, _, ix = (0.5 * h * branch_vector(t - 0.5 * math.pi, 0.0)).astype(int)
    mask[8 + iz, h + 3, 8 + ix] = 1
    
    # Draw main stem 1.
    stem = branch_vector(t, s)
    iz, iy, ix = draw_branch(tree, (8.5, 3, 8.5), stem, h, log)
    mask[iz, iy, ix] = 1
    
    src = (iz, iy - 1, ix)
    t_0 = t
    for j in [0.5, 1.0, 1.2, 1.0]:
        ray = branch_vector(t_0, 0.0)
        t_0 += 0.5 * math.pi
        
        iz, iy, ix = draw_branch(tree, src, ray, 0.2 * j * h, log)
        mask[iz, iy, ix] = 1
    
    
    # Draw main stem 2.
    stem = branch_vector(t + math.pi, s)
    iz, iy, ix = draw_branch(tree, (8.5, 3, 8.5), stem, h, log)
    mask[iz, iy, ix] = 1
    
    src = (iz, iy - 1, ix)
    t_0 = t + math.pi
    for j in [0.5, 1.1, 0.9, 1.0]:
        ray = branch_vector(t_0, 0.0)
        t_0 += 0.5 * math.pi
        
        iz, iy, ix = draw_branch(tree, src, ray, 0.2 * j * h, log)
        mask[iz, iy, ix] = 1
    
    mask = blur(mask.astype(float), max(0.14 * h, 1)) > 0.002
    decay(mask, 0.3, 0.7) # Decay some random leafs on the surface
    
    tree[np.logical_and(mask, tree == 0)] = leaf
    
    trees.append(tree)
    tree_kinds.append("birch")

In [ ]:
visualize_voxels(terrain_color[random.choice(trees)])

#### Define rubber trees

In [ ]:
heights = [
    *[7] * 3,
    *[8] * 9,
    *[12] * 8,
    *[13] * 9,
    *[14] * 7,
]

def rubber_leaf_blur(mask):
    mask = np.roll(mask, 2, axis=1)
    mask[:, 0, :] = False
    return blur(mask, 1)

print(f"Generating {len(heights)} trees...")
for h in heights:    
    tree = np.zeros((16, 32, 16), dtype=int)
    mask = np.zeros((16, 32, 16), dtype=bool)
    
    log = terrain_index["rubber_log"]
    leaf = terrain_index["rubber_leaf"]

    iz, iy, ix = draw_branch(tree, (8, 0, 8), (0, 1, 0), h, log)
    mask[iz, iy, ix] = 1
    
    t = random.random()
    for j in np.linspace(4, 0.8 * h, 4):
        t = t + 0.5 * math.pi
        l1 = np.clip(1 + (h - j) // 4, 3, 6)
        l2 = np.clip(1 + (h - j) // 5, 3, 4)
        
        ray = branch_vector(t, 0.1)
        pos = draw_branch(tree, (8, j, 8), ray, l1, log)
        iz, iy, ix = draw_branch(tree, pos, (0, 1, 0), l2, log)
        mask[iz, iy, ix] = 1
    
    r = 0.17 * h if h <= 10 else 0.1 * h 
    
    mask = blur(2.0 * mask.astype(float), r) > 0.002
    decay(mask, 0.3, 0.7) # Decay some random leafs on the surface
    mask[:, :3, :] = False  # No leaves in bottom 3 voxels
    
    tree[np.logical_and(mask, tree == 0)] = leaf

    trees.append(tree)
    tree_kinds.append("rubber")

In [ ]:
visualize_voxels(terrain_color[random.choice(trees)])

### Generate a map of trees

In [ ]:
MAP_DIM = (2048, 2048)

In [ ]:
@dataclass
class TreeMap:
    trees: List[np.array]
    assignments: np.array

In [ ]:
tm = TreeMap(
    trees=trees,
    assignments=np.zeros(MAP_DIM, dtype=int),
)

In [ ]:
point_mask = np.logical_or(
    periodic_noise(MAP_DIM, 512, [2, 4, 4, 1, 4, 6], seed="tree_groves") > 2,
    periodic_noise(MAP_DIM, 1024, [4, 8, 4, 4, 2, 1], seed="tree_forests") > 2,
)

heatmap_to_image(point_mask.astype(float))

In [ ]:
tree_masks = {
    "oak": point_mask.copy(),
    "birch": np.logical_and(
        point_mask,
        periodic_noise(MAP_DIM, 512, [8, 4, 4, 2, 1], seed="birch_tree") > -1.5,
    ),
    "rubber": np.logical_and(
        point_mask,
        periodic_noise(MAP_DIM, 1024, [4, 8, 4, 4, 2, 1], seed="rubber_tree") > 1.0,
    ),
}

In [ ]:
heatmap_to_image(tree_masks["oak"].astype(float))

In [ ]:
heatmap_to_image(tree_masks["birch"].astype(float))

In [ ]:
heatmap_to_image(tree_masks["rubber"].astype(float))

In [ ]:
spacing = 8

points = np.stack(
    np.meshgrid(
        np.linspace(spacing // 2, MAP_DIM[0] - spacing // 2, MAP_DIM[0] // spacing),
        np.linspace(spacing // 2, MAP_DIM[1] - spacing // 2, MAP_DIM[1] // spacing),
        indexing="ij",
    ),
    axis=-1,
).reshape(-1, 2)

points += 6.0 * np.random.random(points.shape) - 3.0
points = points.astype(int).tolist()

random.shuffle(points)

In [ ]:
%%time

padding = 64
tree_test = np.zeros((MAP_DIM[0] + padding, 32, MAP_DIM[1] + padding), dtype=int)

def placement(x, z, tree):
    t_z, t_y, t_x = tree.shape
    z0 = z - t_z // 2
    x0 = x - t_x // 2
    z1 = z0 + t_z
    x1 = x0 + t_x
    c_z0, c_x0 = max(z0, 0), max(x0, 0)
    c_z1, c_x1 = min(z1, tree_test.shape[0]), min(x1, tree_test.shape[2])
    return (
        c_z0,
        c_x0,
        c_z1,
        c_x1,
        tree[c_z0 - z0:t_z + c_z1 - z1, :, c_x0 - x0:t_x + c_x1 - x1]
    ) 

for z, x in points:
    if z < 0 or x < 0:
        continue
    if z >= MAP_DIM[0] or x >= MAP_DIM[1]:
        continue
    
    # Choose which tree we want to render.
    tree = None
    for i in range(16):
        i = random.randint(0, len(trees) - 1)
        if tree_masks[tree_kinds[i]][z, x]:
            tree = trees[i]
            break
    if tree is None:
        continue
        
    for sz, sx in [(0, 0), (MAP_DIM[0], 0), (0, MAP_DIM[1]), (MAP_DIM[0], MAP_DIM[1])]:
        z0, x0, z1, x1, _ = placement(x, z, tree)
        if np.any(tree_test[z0:z1, :, x0:x1] != 0):
            tree = None
            break
    if tree is None:
        continue
   
    # Place the tree
    for sz, sx in [(0, 0), (MAP_DIM[0], 0), (0, MAP_DIM[1]), (MAP_DIM[0], MAP_DIM[1])]:
        z0, x0, z1, x1, mask = placement(x, z, tree)
        tree_test[z0:z1, :, x0:x1] = mask
    tm.assignments[z, x] = i + 1
    
tree_test = tree_test[:MAP_DIM[0], :, :MAP_DIM[1]]

In [ ]:
visualize_voxels(terrain_color[tree_test[:1024, :32, :1024]])

### Save the tree map.

In [ ]:
with gzip.open("F:/Biomes/alpha_world/features/trees.dump", "wb") as f:
    pickle.dump(tm, f)

### Generate the Flora patterns

In [ ]:
@dataclass
class FloraMap:
    assignments: np.ndarray

In [ ]:
fm = FloraMap(
    assignments=np.zeros(MAP_DIM, dtype=int)
)

In [ ]:
tree_mask = tree_test[:, 0, :] == 0

In [ ]:
grass_mask = np.logical_and(
    tree_mask,
    np.logical_or(
        np.logical_and(
            explicit_noise(MAP_DIM, 64, [2, 4, 4, 1, 4, 6, 4], seed="grass_1") > 3.0,
            explicit_noise(MAP_DIM, 1024, [1, 1, 1, 16, 8, 4, 0, 8], seed="grass_1") > 0.0,
        ),
        explicit_noise(MAP_DIM, 32, [1, 3, 2, 2, 4, 6, 4], seed="grass_2") > 5.0,
    )
)
fm.assignments[grass_mask] = terrain_index["switch_grass"]

In [ ]:
def gen_flower_mask(prevalence, name):
    return np.logical_and(
        grass_mask,
        np.logical_and(
            explicit_noise(MAP_DIM, 16, [4, 1, 4, 6, 8], seed=name) > 8.0,
            explicit_noise(MAP_DIM, 2048, [8, 8, 8, 8, 4, 4, 0, 8], seed=name) > 10 - prevalence,
        ),
    )

In [ ]:
def gen_rare_flower_mask(prevalence, name):
    distro = np.logical_and(
        np.logical_and(
            explicit_noise(MAP_DIM, 128, [1, 0], seed=name) > 0,
            explicit_noise(MAP_DIM, 32, [1, 0], seed=name) > 0.5,
        ),
        np.logical_and(
            explicit_noise(MAP_DIM, 16, [4, 1, 4, 6, 8], seed=name) > 0.0,
            explicit_noise(MAP_DIM, 2048, [0, 0, 16, 32, 4, 4, 0, 8], seed=name) > 20 - prevalence
        )
    )
    return np.logical_and(grass_mask, distro)

In [ ]:
def gen_pumpkin_mask(prevalence, name):
    base = gen_rare_flower_mask(prevalence, name)
    even = np.stack(
        np.meshgrid(np.arange(MAP_DIM[0]), np.arange(MAP_DIM[1])),
        axis=-1,
    )
    even = (even[:, :, 0] + even[:, :, 1]) % 2 == 0
    return np.logical_and(base, even)

In [ ]:
fm.assignments[gen_flower_mask(8, "azalea_flower")] = terrain_index["azalea_flower"]
fm.assignments[gen_flower_mask(8, "bell_flower")] = terrain_index["bell_flower"]
fm.assignments[gen_flower_mask(8, "dandelion_flower")] = terrain_index["dandelion_flower"]
fm.assignments[gen_flower_mask(8, "daylily_flower")] = terrain_index["daylily_flower"]
fm.assignments[gen_flower_mask(8, "lilac_flower")] = terrain_index["lilac_flower"]
fm.assignments[gen_flower_mask(8, "rose_flower")] = terrain_index["rose_flower"]

fm.assignments[gen_rare_flower_mask(2, "cotton_flower")] = terrain_index["cotton_flower"]
fm.assignments[gen_rare_flower_mask(3, "hemp_bush")] = terrain_index["hemp_bush"]
fm.assignments[gen_rare_flower_mask(7, "red_mushroom")] = terrain_index["red_mushroom"]
fm.assignments[gen_pumpkin_mask(16, "pumpkin")] = terrain_index["pumpkin"]

In [ ]:
grass_thinner = np.logical_and(
    explicit_noise(MAP_DIM, 256, [2, 1, 0, 1, 2, 4, 2, 1], seed="grass_patcher") > 2,
    fm.assignments == terrain_index["switch_grass"],
)
fm.assignments[grass_thinner] = 0

In [ ]:
for i in np.unique(fm.assignments):
    name = [name for name, key in terrain_index.items() if key == i][0]
    print(name, (fm.assignments == i).mean(), (fm.assignments == i).sum())

In [ ]:
visualize_voxels(terrain_color[fm.assignments[:, np.newaxis, :]])

In [ ]:
with gzip.open("F:/Biomes/alpha_world/features/florae.dump", "wb") as f:
    pickle.dump(fm, f)